### 0. 데이터 불러오기

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mobeenfatimah/breast-cancer-prediction-dataset-10000-patient")

print("Path to dataset files:", path)

c:\Users\82102\.virtualenvs\study-k17Ljfr1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 234k/234k [00:00<00:00, 1.07MB/s]

Extracting files...
Path to dataset files: C:\Users\82102\.cache\kagglehub\datasets\mobeenfatimah\breast-cancer-prediction-dataset-10000-patient\versions\1


In [2]:
import os

# 다운로드된 폴더 안 파일 목록 확인
for f in os.listdir(path):
    print(f)

breast_cancer_prediction.csv
data_dictionary.csv


In [15]:
import pandas as pd

file_name = "breast_cancer_prediction.csv"
df = pd.read_csv(os.path.join(path, file_name))

df.head()

,Patient_ID,Age,Gender,BMI,Family_History,Smoking,Alcohol_Consumption,Physical_Activity,Hormone_Therapy,Menopause_Status,...,Mammogram_Result,Biopsy_Result,Cancer_Stage,Cancer,Blood_Pressure,Cholesterol,Diabetes,Exercise_Days_Per_Week,Breastfeeding_History,Annual_Income_USD
0,100001,71,Female,27.2,Yes,Yes,Yes,High,No,Post,...,Suspicious,Malignant,Stage II,1,141,167,No,7,No,101316
1,100002,34,Male,26.0,No,No,No,High,No,Post,...,Normal,Benign,No Cancer,0,104,229,Yes,7,Yes,30023
2,100003,80,Female,20.7,No,No,No,High,No,Pre,...,Suspicious,Benign,No Cancer,0,161,275,No,1,Yes,34819
3,100004,40,Female,26.8,Yes,Yes,No,Moderate,No,Post,...,Normal,Benign,No Cancer,0,150,180,No,7,Yes,68025
4,100005,43,Female,28.6,Yes,Yes,No,Moderate,No,Post,...,Normal,Benign,No Cancer,0,110,266,No,0,Yes,90919


In [11]:
df.columns

Index(['Patient_ID', 'Age', 'Gender', 'BMI', 'Family_History', 'Smoking',
       'Alcohol_Consumption', 'Physical_Activity', 'Hormone_Therapy',
       'Menopause_Status', 'Genetic_Mutation', 'Tumor_Size_cm',
       'Lymph_Node_Involvement', 'Mammogram_Result', 'Biopsy_Result',
       'Cancer_Stage', 'Cancer', 'Blood_Pressure', 'Cholesterol', 'Diabetes',
       'Exercise_Days_Per_Week', 'Breastfeeding_History', 'Annual_Income_USD'],
      dtype='str')

### 1. 데이터 전처리

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
# ── 1) 불필요한 컬럼 제거 ────────────────────────────────────────
# - Patient_ID: 식별자, 예측에 무의미
# - Biopsy_Result, Cancer_Stage: Cancer(타겟)와 100% 일치하는 데이터 누수 컬럼
df = df.drop(columns=['Patient_ID', 'Biopsy_Result', 'Cancer_Stage'])

# ── 2) 결측치 처리 ──────────────────────────────────────────────
# 2-1. 결측치 제거 전 암 환자 비율
ratio_before = df['Cancer'].mean()
print(f"결측치 제거 전 암 환자 비율: {ratio_before:.4f} ({ratio_before*100:.2f}%)")
print(f"결측치 제거 전 전체 행 수: {len(df)}")

# 2-2. 결측치가 있는 행 삭제
df_clean = df.dropna()

# 2-3. 결측치 제거 후 암 환자 비율
ratio_after = df_clean['Cancer'].mean()
print(f"결측치 제거 후 암 환자 비율: {ratio_after:.4f} ({ratio_after*100:.2f}%)")
print(f"결측치 제거 후 전체 행 수: {len(df_clean)}")

# ── 3) 범주형 변수 인코딩 ─────────────────────────────────────
# 3-1. 순서형 변수 (Ordinal Encoding)
# Physical_Activity: Low < Moderate < High
activity_order = {'Low': 0, 'Moderate': 1, 'High': 2}
df_clean = df_clean.copy()
df_clean['Physical_Activity'] = df_clean['Physical_Activity'].map(activity_order)

# 3-2. 명목형 변수 (One-Hot Encoding)
# Physical_Activity는 이미 숫자로 변환되어 자동 제외됨
cat_cols = df_clean.select_dtypes(include='object').columns.tolist()
df_encoded = pd.get_dummies(df_clean, columns=cat_cols, drop_first=True)

# ── 4) Train/Test 분리 ──────────────────────────────────────────
X = df_encoded.drop(columns=['Cancer'])
y = df_encoded['Cancer']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── 5) 수치형 변수 스케일링 ──────────────────────────────────
numeric_cols = ['Age', 'BMI', 'Tumor_Size_cm', 'Blood_Pressure', 'Cholesterol',
                 'Exercise_Days_Per_Week', 'Annual_Income_USD', 'Physical_Activity']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

# ── 6) 다중공선성 확인 (VIF) ────────────────────────────────────
X_train_vif = X_train_scaled.astype(float)
vif_df = pd.DataFrame({
    'feature': X_train_vif.columns,
    'VIF': [variance_inflation_factor(X_train_vif.values, i)
            for i in range(X_train_vif.shape[1])]
})
vif_df['VIF'] = vif_df['VIF'].round(2)
display(vif_df.sort_values('VIF', ascending=False))

# ── 7) 최종 데이터셋 확인 ────────────────────────────────────────
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train 분포:\n", y_train.value_counts(normalize=True).round(3))
print("y_test 분포:\n", y_test.value_counts(normalize=True).round(3))

### 2. 모델링(without PCA)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, f1_score)

In [ ]:
# ── 1) 베이스라인 모델 — 로지스틱 회귀 ──────────────────────────
baseline_logit = LogisticRegression(max_iter=1000, random_state=42)
baseline_logit.fit(X_train_scaled, y_train)
baseline_pred = baseline_logit.predict(X_test_scaled)

print("=== 1) Baseline Logistic Regression ===")
print("Accuracy:", accuracy_score(y_test, baseline_pred))
baseline_report = pd.DataFrame(
    classification_report(y_test, baseline_pred, output_dict=True)
).T.round(3)
display(baseline_report)

baseline_cm = pd.DataFrame(
    confusion_matrix(y_test, baseline_pred, labels=baseline_logit.classes_),
    index=[f'실제_{c}' for c in baseline_logit.classes_],
    columns=[f'예측_{c}' for c in baseline_logit.classes_]
)
display(baseline_cm)

# ── 2) Decision Tree ─────────────────────────────────────────────
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train_scaled, y_train)
dt_pred = dt.predict(X_test_scaled)

print("\n=== 2) Decision Tree ===")
print("Accuracy:", accuracy_score(y_test, dt_pred))
dt_report = pd.DataFrame(
    classification_report(y_test, dt_pred, output_dict=True)
).T.round(3)
display(dt_report)

dt_cm = pd.DataFrame(
    confusion_matrix(y_test, dt_pred, labels=dt.classes_),
    index=[f'실제_{c}' for c in dt.classes_],
    columns=[f'예측_{c}' for c in dt.classes_]
)
display(dt_cm)

dt_importance = pd.Series(
    dt.feature_importances_, index=X_train_scaled.columns
).sort_values(ascending=False).to_frame('importance')
display(dt_importance)

# ── 3) Gradient Boosting ─────────────────────────────────────────
gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train_scaled, y_train)
gb_pred = gb.predict(X_test_scaled)

print("\n=== 3) Gradient Boosting ===")
print("Accuracy:", accuracy_score(y_test, gb_pred))
gb_report = pd.DataFrame(
    classification_report(y_test, gb_pred, output_dict=True)
).T.round(3)
display(gb_report)

gb_cm = pd.DataFrame(
    confusion_matrix(y_test, gb_pred, labels=gb.classes_),
    index=[f'실제_{c}' for c in gb.classes_],
    columns=[f'예측_{c}' for c in gb.classes_]
)
display(gb_cm)

gb_importance = pd.Series(
    gb.feature_importances_, index=X_train_scaled.columns
).sort_values(ascending=False).to_frame('importance')
display(gb_importance)

# ── 4) 두 모델 변수 중요도 비교 ───────────────────────────────
imp_compare_df = pd.DataFrame({
    'DT_Importance': dt_importance['importance'],
    'GB_Importance': gb_importance['importance']
}).round(4).sort_values('GB_Importance', ascending=False).reset_index()
imp_compare_df.columns = ['Feature', 'DT_Importance', 'GB_Importance']

print("\n=== 4) Decision Tree vs Gradient Boosting 변수 중요도 비교 ===")
display(imp_compare_df)

# ── 5) 교차검증 — Stratified K-Fold ──────────────────────────────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}
for name, model in [
    ('Logistic', LogisticRegression(max_iter=1000, random_state=42)),
    ('DecisionTree', DecisionTreeClassifier(random_state=42)),
    ('GradientBoosting', GradientBoostingClassifier(random_state=42))
]:
    scores = []
    for train_idx, val_idx in skf.split(X_train_scaled, y_train):
        X_tr, X_val = X_train_scaled.iloc[train_idx], X_train_scaled.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        model.fit(X_tr, y_tr)
        pred = model.predict(X_val)
        scores.append(f1_score(y_val, pred, average='macro'))
    cv_results[name] = scores

cv_summary = pd.DataFrame(cv_results)
print("\n=== 6) Cross-Validation (5-Fold, Macro F1) ===")
display(cv_summary)
display(cv_summary.mean().round(3).to_frame('mean_macro_f1'))

# ── 6) 모델별 성능 비교 (Test set 기준) ──────────────────────────
comparison_full = pd.DataFrame({
    'Accuracy': [
        accuracy_score(y_test, baseline_pred),
        accuracy_score(y_test, dt_pred),
        accuracy_score(y_test, gb_pred)
    ],
    'Macro F1': [
        f1_score(y_test, baseline_pred, average='macro'),
        f1_score(y_test, dt_pred, average='macro'),
        f1_score(y_test, gb_pred, average='macro')
    ],
    'Weighted F1': [
        f1_score(y_test, baseline_pred, average='weighted'),
        f1_score(y_test, dt_pred, average='weighted'),
        f1_score(y_test, gb_pred, average='weighted')
    ]
}, index=['Logistic Regression', 'Decision Tree', 'Gradient Boosting']).round(3)

print("\n=== 7) 모델별 최종 성능 비교 (PCA 전) ===")
display(comparison_full)

### 3. 모델링(with PCA)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

In [ ]:
# ── 1) PCA — 주성분 개수 결정 (설명분산 95% 기준) ──────────────────
pca_full = PCA(random_state=42)
pca_full.fit(X_train_scaled)

explained_var = pca_full.explained_variance_ratio_
cum_var = np.cumsum(explained_var)
n_components_95 = np.argmax(cum_var >= 0.95) + 1
print(f"95% 설명분산 도달 주성분 개수: {n_components_95}")

evr_df = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(explained_var))],
    "explained_variance_ratio": explained_var,
    "cumulative_variance_ratio": cum_var,
})
display(evr_df.round(4))

# ── 2) Scree Plot & 누적 설명분산 그래프 ────────────────────────
plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows 환경 기준, 필요 시 조정
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, len(explained_var) + 1), explained_var, alpha=0.7, color='steelblue')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot')
axes[0].set_xticks(range(1, len(explained_var) + 1))

axes[1].plot(range(1, len(cum_var) + 1), cum_var, marker='o', color='darkorange')
axes[1].axhline(y=0.95, color='red', linestyle='--', label='95% 기준선')
axes[1].axvline(x=n_components_95, color='green', linestyle='--',
                label=f'선택된 주성분 수: {n_components_95}')
axes[1].set_xlabel('Number of Principal Components')
axes[1].set_ylabel('Cumulative Explained Variance Ratio')
axes[1].set_title('Cumulative Explained Variance')
axes[1].set_xticks(range(1, len(cum_var) + 1))
axes[1].legend()

plt.tight_layout()
plt.show()

# ── 3) 최종 PCA 적용 (16개 주성분 고정) ─────────────────────────
pca = PCA(n_components=16, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)   # test는 transform만 (누수 방지)

pc_cols = [f"PC{i+1}" for i in range(16)]
X_train_pca = pd.DataFrame(X_train_pca, columns=pc_cols, index=X_train_scaled.index)
X_test_pca = pd.DataFrame(X_test_pca, columns=pc_cols, index=X_test_scaled.index)

print("X_train_pca shape:", X_train_pca.shape)
print("X_test_pca shape:", X_test_pca.shape)
print(f"16개 주성분 누적 설명분산: {pca.explained_variance_ratio_.sum():.4f}")

# ── 4) 베이스라인 모델 — 로지스틱 회귀 (PCA) ────────────────────
logit = LogisticRegression(max_iter=1000, random_state=42)
logit.fit(X_train_pca, y_train)
logit_pred = logit.predict(X_test_pca)

print("\n=== a. Logistic Regression (PCA) ===")
print("Accuracy:", accuracy_score(y_test, logit_pred))
logit_report = pd.DataFrame(
    classification_report(y_test, logit_pred, output_dict=True)
).T.round(3)
display(logit_report)

logit_cm = pd.DataFrame(
    confusion_matrix(y_test, logit_pred, labels=logit.classes_),
    index=[f'실제_{c}' for c in logit.classes_],
    columns=[f'예측_{c}' for c in logit.classes_]
)
display(logit_cm)

# ── 5) Decision Tree (PCA) ───────────────────────────────────────
dt_pca = DecisionTreeClassifier(random_state=42)
dt_pca.fit(X_train_pca, y_train)
dt_pca_pred = dt_pca.predict(X_test_pca)

print("\n=== b. Decision Tree (PCA) ===")
print("Accuracy:", accuracy_score(y_test, dt_pca_pred))
dt_pca_report = pd.DataFrame(
    classification_report(y_test, dt_pca_pred, output_dict=True)
).T.round(3)
display(dt_pca_report)

dt_pca_cm = pd.DataFrame(
    confusion_matrix(y_test, dt_pca_pred, labels=dt_pca.classes_),
    index=[f'실제_{c}' for c in dt_pca.classes_],
    columns=[f'예측_{c}' for c in dt_pca.classes_]
)
display(dt_pca_cm)

dt_pca_importance = pd.Series(
    dt_pca.feature_importances_, index=X_train_pca.columns
).sort_values(ascending=False).to_frame('importance')
display(dt_pca_importance)

# ── 6) Gradient Boosting (PCA) ───────────────────────────────────
gb_pca = GradientBoostingClassifier(random_state=42)
gb_pca.fit(X_train_pca, y_train)
gb_pca_pred = gb_pca.predict(X_test_pca)

print("\n=== c. Gradient Boosting (PCA) ===")
print("Accuracy:", accuracy_score(y_test, gb_pca_pred))
gb_pca_report = pd.DataFrame(
    classification_report(y_test, gb_pca_pred, output_dict=True)
).T.round(3)
display(gb_pca_report)

gb_pca_cm = pd.DataFrame(
    confusion_matrix(y_test, gb_pca_pred, labels=gb_pca.classes_),
    index=[f'실제_{c}' for c in gb_pca.classes_],
    columns=[f'예측_{c}' for c in gb_pca.classes_]
)
display(gb_pca_cm)

gb_pca_importance = pd.Series(
    gb_pca.feature_importances_, index=X_train_pca.columns
).sort_values(ascending=False).to_frame('importance')
display(gb_pca_importance)

# ── 7) 두 모델 변수 중요도 비교 (PCA) ────────────────────────────
imp_compare_pca_df = pd.DataFrame({
    'DT_Importance': dt_pca_importance['importance'],
    'GB_Importance': gb_pca_importance['importance']
}).round(4).sort_values('GB_Importance', ascending=False).reset_index()
imp_compare_pca_df.columns = ['Feature', 'DT_Importance', 'GB_Importance']

print("\n=== d. Decision Tree vs Gradient Boosting 변수 중요도 비교 (PCA) ===")
display(imp_compare_pca_df)

# ── 8) 교차검증 — Stratified K-Fold (PCA) ────────────────────────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results_pca = {}
for name, model in [
    ('Logistic', LogisticRegression(max_iter=1000, random_state=42)),
    ('DecisionTree', DecisionTreeClassifier(random_state=42)),
    ('GradientBoosting', GradientBoostingClassifier(random_state=42))
]:
    scores = []
    for train_idx, val_idx in skf.split(X_train_pca, y_train):
        X_tr, X_val = X_train_pca.iloc[train_idx], X_train_pca.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        model.fit(X_tr, y_tr)
        pred = model.predict(X_val)
        scores.append(f1_score(y_val, pred, average='macro'))
    cv_results_pca[name] = scores

cv_summary_pca = pd.DataFrame(cv_results_pca)
print("\n=== 교차검증 (5-Fold, Macro F1, PCA) ===")
display(cv_summary_pca)
display(cv_summary_pca.mean().round(3).to_frame('mean_macro_f1'))

# ── 9) 모델별 성능 비교 (Test set 기준, PCA) ─────────────────────
comparison_pca = pd.DataFrame({
    'Accuracy': [
        accuracy_score(y_test, logit_pred),
        accuracy_score(y_test, dt_pca_pred),
        accuracy_score(y_test, gb_pca_pred)
    ],
    'Macro F1': [
        f1_score(y_test, logit_pred, average='macro'),
        f1_score(y_test, dt_pca_pred, average='macro'),
        f1_score(y_test, gb_pca_pred, average='macro')
    ],
    'Weighted F1': [
        f1_score(y_test, logit_pred, average='weighted'),
        f1_score(y_test, dt_pca_pred, average='weighted'),
        f1_score(y_test, gb_pca_pred, average='weighted')
    ]
}, index=['Logistic Regression', 'Decision Tree', 'Gradient Boosting']).round(3)

print("\n=== 모델별 최종 성능 비교 (PCA) ===")
display(comparison_pca)

### 4. PCA 전 후 비교

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# ── 원본 변수 vs PCA 결과 비교 (위에서 계산된 comparison_full, comparison_pca 사용) ──
print("PCA 전 (원본 변수)")
display(comparison_full)
print("PCA 후 (16개 주성분)")
display(comparison_pca)

# ── 모델별 그래프 (3개 서브플롯) ──────────────────────────────
metrics = ['Accuracy', 'Macro F1', 'Weighted F1']
model_names = ['Logistic Regression', 'Decision Tree', 'Gradient Boosting']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, model in enumerate(model_names):
    before = comparison_full.loc[model, metrics].values
    after = comparison_pca.loc[model, metrics].values

    x = np.arange(len(metrics))
    width = 0.35

    axes[i].bar(x - width/2, before, width, label='PCA 전', color='#4C72B0')
    axes[i].bar(x + width/2, after, width, label='PCA 후', color='#DD8452')

    axes[i].set_title(model, fontsize=13)
    axes[i].set_xticks(x)
    axes[i].set_xticklabels(metrics)
    axes[i].set_ylim(0, 1)
    axes[i].legend()

    # 값 라벨 표시
    for j, (b, a) in enumerate(zip(before, after)):
        axes[i].text(j - width/2, b + 0.02, f'{b:.3f}', ha='center', fontsize=9)
        axes[i].text(j + width/2, a + 0.02, f'{a:.3f}', ha='center', fontsize=9)

plt.suptitle('모델별 PCA 적용 전/후 성능 비교', fontsize=15)
plt.tight_layout()
plt.show()